In [38]:
import pandas as pd
import numpy as np

In [39]:
df= pd.read_csv("/content/IMDB Dataset.csv")
df.shape

(50000, 2)

In [40]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [41]:
df['sentiment'].value_counts()

,count
sentiment,
positive,25000
negative,25000


In [42]:
df['review'][7]

"This show was an amazing, fresh & innovative idea in the 70's when it first aired. The first 7 or 8 years were brilliant, but things dropped off after that. By 1990, the show was not really funny anymore, and it's continued its decline further to the complete waste of time it is today.<br /><br />It's truly disgraceful how far this show has fallen. The writing is painfully bad, the performances are almost as bad - if not for the mildly entertaining respite of the guest-hosts, this show probably wouldn't still be on the air. I find it so hard to believe that the same creator that hand-selected the original cast also chose the band of hacks that followed. How can one recognize such brilliance and then see fit to replace it with such mediocrity? I felt I must give 2 stars out of respect for the original cast that made this show such a huge success. As it is now, the show is just awful. I can't believe it's still on the air."

In [43]:
df['positive_review']= df['sentiment'].apply(lambda x: 1 if x=='positive' else 0)
df.head()

,review,sentiment,positive_review
0,One of the other reviewers has mentioned that ...,positive,1
1,A wonderful little production. <br /><br />The...,positive,1
2,I thought this was a wonderful way to spend ti...,positive,1
3,Basically there's a family where a little boy ...,negative,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,1


In [44]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 106.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [45]:
import spacy
nlp= spacy.load("en_core_web_sm")

In [46]:
def cleaned(doc):
    useful = []
    for token in doc:
        if token.tag_ not in ['X', 'PUNCT']:
            useful.append(token.text)
    return " ".join(useful)

In [47]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test= train_test_split(df.review, df.positive_review, test_size= 0.2)

In [48]:
X_train.shape

(40000,)

In [49]:
X_test.shape

(10000,)

In [50]:
from sklearn.feature_extraction.text import CountVectorizer
v= CountVectorizer()

In [51]:
X_train_cv= v.fit_transform(X_train.values)

In [52]:
X_train_cv.shape

(40000, 93223)

In [53]:
v.get_feature_names_out().shape

(93223,)

##Exercise-1 Solution

In [54]:
X_test_cv= v.transform(X_test.values)

In [55]:
v.get_feature_names_out().shape

(93223,)

In [56]:
from sklearn.ensemble import RandomForestClassifier

In [57]:
model= RandomForestClassifier(n_estimators= 50, criterion= 'entropy')

In [58]:
model.fit(X_train_cv, y_train)

RandomForestClassifier(criterion='entropy', n_estimators=50)

In [59]:
yp_forest= model.predict(X_test_cv)

In [60]:
from sklearn.metrics import classification_report
print(classification_report(y_test, yp_forest))

              precision    recall  f1-score   support

           0       0.84      0.84      0.84      5020
           1       0.84      0.84      0.84      4980

    accuracy                           0.84     10000
   macro avg       0.84      0.84      0.84     10000
weighted avg       0.84      0.84      0.84     10000



##Exercise-2 Solution

In [61]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline

knn_pipe= Pipeline([
    ('vectorizer', CountVectorizer()),
    ('knn', KNeighborsClassifier())
    ])

In [62]:
knn_pipe.fit(X_train, y_train)

Pipeline(steps=[('vectorizer', CountVectorizer()),
                ('knn', KNeighborsClassifier())])

In [63]:
yp_knn= knn_pipe.predict(X_test)
print(classification_report(y_test, yp_knn))

              precision    recall  f1-score   support

           0       0.66      0.57      0.61      5020
           1       0.62      0.70      0.65      4980

    accuracy                           0.63     10000
   macro avg       0.64      0.63      0.63     10000
weighted avg       0.64      0.63      0.63     10000



##Exercise-3 Solution

In [64]:
from sklearn.naive_bayes import MultinomialNB
nb_pipe= Pipeline([
    ('vectorizer', CountVectorizer()),
    ('nb', MultinomialNB())
    ])
nb_pipe.fit(X_train, y_train)
yp_nb= nb_pipe.predict(X_test)
print(classification_report(y_test, yp_nb))

              precision    recall  f1-score   support

           0       0.83      0.88      0.86      5020
           1       0.87      0.82      0.85      4980

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85      0.85     10000

